# G07 · A2 Task 1 — Score OCR against your own hand-typed ground truth (printed pp. 16–29)

You've transcribed printed pages **16–29** yourself into one `.txt` file per page. This
notebook finds where those printed page numbers actually sit in the PDF (they are **not**
guaranteed to match the PDF's own page order — 1890s books often number front matter with
roman numerals or leave early leaves unnumbered), loads your files, writes the repo's
`grading_kit/heldout_pages/` + `labels.jsonl`, and scores the default Tesseract pipeline
against your text (CER / WER / word-F1).

## Before you run
1. **Upload your 14 `.txt` files as a Kaggle dataset**, one file per page, named however you
   like as long as the printed page number appears in the filename (e.g. `16.txt`, `page_16.txt`,
   `pg16.txt` all work — the notebook extracts the number and prints a confirmation table for
   you to check before anything is scored).
2. **Attach that dataset** (Add Input) — no other sidecar dataset is needed for this notebook.
3. Internet **ON** (for the repo clone + tesseract install; the PDF is fetched from Internet
   Archive if you don't already have it as a Kaggle input).

## Why there's a whole "locate the pages" section below
Printed page 16 could be the PDF's 16th image, or its 40th, or it could be inside a
roman-numeral preface (xvi) rather than the main Arabic-numbered body — there is no safe way
to guess this. The notebook reads the scan's own embedded OCR text layer to **propose** which
PDF pages carry printed "16" and "29" (checking both Arabic and Roman numerals), then shows
you the actual rendered images so you confirm by eye before anything downstream depends on it.
**Do not skip the confirmation step** — a wrong anchor silently mislabels all 14 pages.

*Note: a contiguous 14-page block from one part of the book won't include a figure/table/
landscape page — worth adding 1–2 more pages later from elsewhere in the book if you want
fuller coverage, but not required to finish this pass.*


In [ ]:
# ── 1. Environment ────────────────────────────────────────────────────────────
import os, sys, subprocess, time, json, re

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if subprocess.run(["which", "tesseract"], capture_output=True).returncode != 0:
    sh("apt-get -qq update > /dev/null 2>&1 || true")
    sh("apt-get -qq install -y tesseract-ocr > /dev/null 2>&1")

%pip install -q "pymupdf>=1.25.5,<1.26" "opencv-python-headless>=4.10,<5.0" "pytesseract>=0.3.13,<0.4" "pydantic>=2.7,<3.0" "pydantic-settings>=2.2,<3.0" "pyyaml>=6.0,<7.0"

REPO_URL = "https://github.com/smammahdi/doc-agent-G07.git"
PIN = "45c3fc3"
if not os.path.exists("/kaggle/working/repo"):
    sh(f"git clone -q {REPO_URL} /kaggle/working/repo")
sh(f"cd /kaggle/working/repo && git checkout -q {PIN} && git log --oneline -1")
sys.path.insert(0, "/kaggle/working/repo/src")

import fitz, cv2, pytesseract  # noqa: E402
TESSERACT_VERSION = subprocess.run(["tesseract", "--version"], capture_output=True, text=True).stdout.splitlines()[0]
print("tesseract:", TESSERACT_VERSION, "| pymupdf:", getattr(fitz, "__version__", None) or fitz.version[0])


In [ ]:
# ── 2. PDF: reuse your uploaded copy if present, else fetch + verify from IA ──
from pathlib import Path
import hashlib, urllib.request

CANDIDATE = Path("/kaggle/input/datasets/kmazd1110/dl-peoples-common-sense-med-advisor/"
                  "EN_The-Peoples-Common-Sense-Medical-Adviser.pdf")
EXPECTED_BYTES = 65311598
EXPECTED_SHA = "841b1feb55ff0aff5735c3aeb308eb52e217f91ae55c5d34e21feb6a640c8896"

def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

if CANDIDATE.exists():
    PDF = CANDIDATE
    print("using uploaded copy:", PDF)
else:
    RAW = Path("/kaggle/working/data/raw"); RAW.mkdir(parents=True, exist_ok=True)
    PDF = RAW / "pierce-peoples-common-sense-medical-adviser-1890.pdf"
    if not (PDF.exists() and PDF.stat().st_size == EXPECTED_BYTES and sha256(PDF) == EXPECTED_SHA):
        print("uploaded copy not found — downloading ~65MB from Internet Archive instead...")
        urllib.request.urlretrieve(
            "https://archive.org/download/peoplescommonsen00pier/peoplescommonsen00pier.pdf", PDF)
    assert sha256(PDF) == EXPECTED_SHA, "downloaded PDF failed hash verification"
    print("using downloaded+verified copy:", PDF)

doc = fitz.open(str(PDF))
print(f"PDF has {len(doc)} pages total")


## Step A — locate printed pages 16 and 29 in the PDF (do not skip)

In [ ]:
# ── 3. Auto-suggest candidates from the embedded text layer ───────────────────
ROMAN_MAP = {"i": 1, "v": 5, "x": 10, "l": 50, "c": 100, "d": 500, "m": 1000}

def roman_to_int(s):
    s = s.lower()
    if not s or any(c not in ROMAN_MAP for c in s):
        return None
    total = 0
    for i, c in enumerate(s):
        v = ROMAN_MAP[c]
        if i + 1 < len(s) and ROMAN_MAP[s[i + 1]] > v:
            total -= v
        else:
            total += v
    return total if total > 0 else None

def guess_numbers_on_page(text):
    found = {}  # number -> "arabic" | "roman"
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    for l in (lines[:3] + lines[-3:]):
        core = l.strip("[](){}.,;-— ")
        if core.isdigit() and len(core) <= 4:
            found[int(core)] = "arabic"
        else:
            r = roman_to_int(core)
            if r is not None and r <= 200:
                found[r] = "roman"
    return found

TARGETS = {16, 29}
GRID_END = min(80, len(doc))  # generous front-matter search window; extend if not found below
hits = {t: [] for t in TARGETS}
for i in range(GRID_END):
    nums = guess_numbers_on_page(doc[i].get_text())
    for t in TARGETS:
        if t in nums:
            hits[t].append((i + 1, nums[t]))  # 1-indexed PDF page, numeral system

for t in TARGETS:
    print(f"printed page {t}: candidate PDF page(s) = {hits[t]}")
if not hits[16] and not hits[29]:
    print("\nNo candidates found via embedded text (scan may lack a text layer, or "
          "GRID_END is too small). Falling back to a visual scan in the next cell — "
          "widen GRID_END above and rerun if needed.")


In [ ]:
# ── 4. Visual confirmation: render the candidates (or a manual range) full-size ──
import matplotlib.pyplot as plt

def show_pdf_page(pdf_index_1based, dpi=120, width=6, label=""):
    pix = doc[pdf_index_1based - 1].get_pixmap(dpi=dpi)
    import numpy as np
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    plt.figure(figsize=(width, width * pix.height / pix.width))
    plt.imshow(img if pix.n == 3 else img[:, :, :3])
    plt.axis("off")
    plt.title(f"PDF page {pdf_index_1based}" + (f"  ({label})" if label else ""))
    plt.show()

# Show every distinct candidate PDF page from cell 3. If nothing was found automatically,
# set MANUAL_SCAN_RANGE = range(1, 41) (or wider) to eyeball pages by hand instead.
MANUAL_SCAN_RANGE = None

candidates = sorted({p for t in TARGETS for p, _ in hits[t]})
if candidates:
    for p in candidates:
        show_pdf_page(p, label=f"candidate for printed 16 or 29")
elif MANUAL_SCAN_RANGE:
    for p in MANUAL_SCAN_RANGE:
        show_pdf_page(p, dpi=80, width=4)
else:
    print("No candidates and MANUAL_SCAN_RANGE not set — set it above (e.g. range(1, 41)) and rerun.")


In [ ]:
# ── 5. Set the confirmed anchor after looking at the images above ─────────────
# The PDF page number (1-indexed, matching the images in cell 4) that shows PRINTED "16".
ANCHOR_PDF_PAGE_FOR_PRINTED_16 = None  # ← fill this in after confirming visually

assert ANCHOR_PDF_PAGE_FOR_PRINTED_16 is not None, (
    "Set ANCHOR_PDF_PAGE_FOR_PRINTED_16 above to the PDF page number you confirmed shows "
    "printed page 16, then rerun this cell."
)
# Assumes printed 16..29 are 14 CONSECUTIVE pdf pages (no unnumbered plate inserted mid-range —
# check the full render in the next cell to be sure).
printed_to_pdf = {n: ANCHOR_PDF_PAGE_FOR_PRINTED_16 + (n - 16) for n in range(16, 30)}
printed_to_id = {n: f"p{pdf_idx:04d}" for n, pdf_idx in printed_to_pdf.items()}
print("printed page -> PDF page -> page_id")
for n in range(16, 30):
    print(f"  {n:>3} -> {printed_to_pdf[n]:>4} -> {printed_to_id[n]}")


In [ ]:
# ── 6. Final render of all 14 pages — last chance to confirm before scoring ───
for n in range(16, 30):
    show_pdf_page(printed_to_pdf[n], dpi=150, width=6, label=f"printed page {n}")
print("Do these 14 images match what you transcribed, in order, with no inserted plate "
      "skipped? If yes, continue. If not, fix ANCHOR_PDF_PAGE_FOR_PRINTED_16 in cell 5.")


## Step B — load your ground-truth text and score Tesseract

In [ ]:
# ── 7. Auto-discover your uploaded .txt files and map them to page_ids ────────
txt_files = []
for root, _, files in os.walk("/kaggle/input"):
    for name in files:
        if name.lower().endswith(".txt"):
            txt_files.append(os.path.join(root, name))

def extract_page_number(filename):
    nums = re.findall(r"\d+", filename)
    for n in nums:
        if 16 <= int(n) <= 29:
            return int(n)
    return None

# If auto-detection gets any file wrong, fix it here: {"filename.txt": printed_page_number}
MANUAL_MAP = {}

mapping = {}  # printed_page_number -> filepath
for f in txt_files:
    name = os.path.basename(f)
    n = MANUAL_MAP.get(name) or extract_page_number(name)
    if n is not None:
        mapping[n] = f

print(f"{'printed page':>13}  filename")
for n in range(16, 30):
    print(f"{n:>13}  {os.path.basename(mapping[n]) if n in mapping else 'MISSING'}")
missing = [n for n in range(16, 30) if n not in mapping]
assert not missing, f"no .txt file matched for printed page(s) {missing} — fix filenames or MANUAL_MAP above"


In [ ]:
# ── 8. Render page images + write grading_kit/heldout_pages + labels.jsonl ───
from doc_agent.ingest import loader  # noqa: E402

OUT = Path("/kaggle/working/out")
HELDOUT = OUT / "grading_kit" / "heldout_pages"; HELDOUT.mkdir(parents=True, exist_ok=True)

labels = []
for n in range(16, 30):
    pid = printed_to_id[n]
    target = HELDOUT / f"{pid}.jpg"
    if not target.exists():
        loader._atomic_render(doc[printed_to_pdf[n] - 1], target, 300, 80)
    text = Path(mapping[n]).read_text(encoding="utf-8")
    labels.append({"page_id": pid, "text": text})

labels_path = OUT / "grading_kit" / "labels.jsonl"
with open(labels_path, "w", encoding="utf-8") as f:
    for row in labels:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"wrote {len(labels)} pages to {HELDOUT} and {labels_path}")
for row in labels:
    print(f"  {row['page_id']}: {len(row['text'])} chars")


In [ ]:
# ── 9. Run the repo's DEFAULT pipeline (projection layout + Tesseract) ────────
from doc_agent import config as config_mod  # noqa: E402
from doc_agent.vision import layout as layout_mod, ocr as ocr_mod  # noqa: E402
from doc_agent.contracts import Page  # noqa: E402

cfg = config_mod.load("/kaggle/working/repo/configs/config.yaml")
cfg["page_images"] = {row["page_id"]: str(HELDOUT / f"{row['page_id']}.jpg") for row in labels}
pages = [Page(id=row["page_id"], image_path=cfg["page_images"][row["page_id"]], doc_id="pierce-1890")
         for row in labels]

regions = layout_mod.detect(pages, cfg)
chunks = ocr_mod.transcribe(regions, cfg)
hyp = {row["page_id"]: "" for row in labels}
for ch in chunks:
    hyp[ch.page_ids[0]] = (hyp[ch.page_ids[0]] + " " + ch.text).strip()
print("Tesseract OCR complete on", len(labels), "pages")


In [ ]:
# ── 10. Score: CER / WER / word-F1 against your hand-typed labels ─────────────
from collections import Counter

def norm(s):
    return re.sub(r"\s+", " ", s).strip()

def lev(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[-1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def word_f1(h, r):
    H, R = Counter(h), Counter(r)
    tp = sum((H & R).values())
    if not h and not r:
        return 1.0
    if tp == 0:
        return 0.0
    p, rc = tp / sum(H.values()), tp / sum(R.values())
    return 2 * p * rc / (p + rc)

page_cer = {}
tot_ce = tot_we = tot_chars = tot_words = 0
per_page = []
print(f"{'page':7} {'printed':>7} {'ref-chars':>9} {'CER':>7} {'WER':>7} {'wordF1':>7}")
for n, row in zip(range(16, 30), labels, strict=True):
    pid = row["page_id"]
    ref, h = norm(row["text"]), norm(hyp[pid])
    rw, hw = ref.split(), h.split()
    ce, we = lev(h, ref), lev(hw, rw)
    cer = ce / len(ref) if ref else (0.0 if not h else float("inf"))
    wer = we / len(rw) if rw else (0.0 if not hw else float("inf"))
    f1 = word_f1(hw, rw)
    page_cer[pid] = cer
    per_page.append(dict(page_id=pid, printed_page=n, ref_chars=len(ref), cer=cer, wer=wer, word_f1=f1))
    tot_ce += ce; tot_we += we; tot_chars += len(ref); tot_words += len(rw)
    print(f"{pid:7} {n:>7} {len(ref):>9} {cer:>7.3f} {wer:>7.3f} {f1:>7.3f}")

AGG = dict(micro_cer=tot_ce / max(1, tot_chars), micro_wer=tot_we / max(1, tot_words),
           n_pages=len(labels), ref_chars=tot_chars, ref_words=tot_words)
print(f"\nAGGREGATE (micro): CER={AGG['micro_cer']:.4f}  WER={AGG['micro_wer']:.4f}  "
      f"over {AGG['n_pages']} pages, {tot_chars:,} ref chars / {tot_words:,} ref words")

with open(OUT / "metrics_tesseract.json", "w") as f:
    json.dump(dict(engine=TESSERACT_VERSION, layout="projection (repo default)", commit=PIN,
                   anchor_pdf_page_for_printed_16=ANCHOR_PDF_PAGE_FOR_PRINTED_16,
                   aggregate=AGG, per_page=per_page), f, indent=2)
print("saved: out/metrics_tesseract.json")


In [ ]:
# ── 11. Worst failure (for the report) ─────────────────────────────────────────
worst_pid = max(page_cer, key=lambda p: (page_cer[p] if page_cer[p] != float("inf") else 9e9))
worst_n = [n for n in range(16, 30) if printed_to_id[n] == worst_pid][0]
print(f"worst page: printed {worst_n} ({worst_pid}), CER={page_cer[worst_pid]:.3f}")
show_pdf_page(printed_to_pdf[worst_n], dpi=150, width=6, label=f"printed page {worst_n} — worst CER")
ref_text = [r["text"] for r in labels if r["page_id"] == worst_pid][0]
print("── YOUR GROUND TRUTH ──"); print(norm(ref_text)[:600])
print("── TESSERACT ──");        print(norm(hyp[worst_pid])[:600])


In [ ]:
# ── 12. Package for download ───────────────────────────────────────────────────
import shutil
zip_path = shutil.make_archive("/kaggle/working/heldout_score_output", "zip", str(OUT))
print("DONE. Download from the right panel → Output: heldout_score_output.zip")
print("  contains: grading_kit/heldout_pages/*.jpg, grading_kit/labels.jsonl, metrics_tesseract.json")
print()
print(json.dumps(AGG, indent=2))
print("\nHand this zip back to Claude to commit grading_kit/heldout_pages/ + labels.jsonl into the repo.")
